
🛠️ 1. 콘솔 출력 캡처하기
YOLO 실행 시 출력되는 로그를 파이썬 코드에서 문자열로 저장할 수 있어요.


In [ ]:

import subprocess

result = subprocess.run(
    ['yolo', 'task=val', 'mode=val', 'model=best.pt'],  # YOLO 명령어 예시
    capture_output=True,
    text=True
)

console_output = result.stdout



이렇게 하면 console_output 변수에 YOLO 실행 결과가 문자열로 저장돼요.

🗃️ 2. DB에 저장하기
이제 이 문자열을 원하는 DB에 저장하면 됩니다. 예를 들어 SQLite를 사용한다면:

In [ ]:
import sqlite3

conn = sqlite3.connect('yolo_results.db')
cursor = conn.cursor()

# 테이블 생성
cursor.execute('''
    CREATE TABLE IF NOT EXISTS yolo_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        log TEXT
    )
''')

# 로그 저장
cursor.execute('INSERT INTO yolo_logs (log) VALUES (?)', (console_output,))
conn.commit()
conn.close()


3. 구조화된 데이터로 저장하고 싶다면?

단순한 로그 문자열이 아니라, 각 클래스별 성능 지표(Box Precision, Recall, mAP 등)를 파싱해서 구조화된 형태로 저장할 수도 있어요. 
예를 들어 정규표현식이나 문자열 분할을 활용해서 다음과 같이 처리할 수 있죠:

In [ ]:
import re

pattern = r'(\w[\w-]*)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)'
matches = re.findall(pattern, console_output)

for cls, img, inst, box_p, recall, map50, map5095 in matches:
    cursor.execute('''
        INSERT INTO yolo_metrics (class, images, instances, box_precision, recall, map50, map5095)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (cls, img, inst, box_p, recall, map50, map5095))


💡 팁
DB는 MySQL, PostgreSQL, MongoDB 등 원하는 걸로 바꿔서 사용 가능해요.

로그 저장 외에도, 모델 버전, 실행 환경(GPU, PyTorch 버전 등)도 함께 저장하면 추후 분석에 유용해요.